# Exercise 4 (MapReduce in Practice) [3 points]
---

For this exercise, you are tasked with writing your own MapReduce program in Python and to
run it on the cluster on the provided datasets.   
You may look at the exercise sheet for all the information on the datasets and this task.

In [1]:
# Save variables to access the file locations
# posts='/home/adbs_shared/Ex_2/stackexchange/posts.csv' # cluster
posts='stackexchange/posts.csv' # local

# users='/home/adbs_shared/Ex_2/stackexchange/users.csv'
users='stackexchange/users.csv'

# comments='/home/adbs_shared/Ex_2/stackexchange/comments.csv'
comments='stackexchange/comments.csv'


In [2]:
# %%bash
# pip3 install mrjob mr3px six

# Load data to check

In [3]:
import pandas as pd

post_df = pd.read_csv("stackexchange/posts.csv")
comment_df = pd.read_csv("stackexchange/comments.csv")
user_df = pd.read_csv("stackexchange/users.csv")

print(post_df.shape)
print(comment_df.shape)
print(user_df.shape)

print(post_df.columns)
print(comment_df.columns)
print(user_df.columns)

(309024, 20)
(568838, 6)
(227032, 10)
Index(['id', 'acceptedanswerid', 'answercount', 'closeddate', 'commentcount',
       'communityowneddate', 'creationdate', 'favoritecount',
       'lastactivitydate', 'lasteditdate', 'lasteditordisplayname',
       'lasteditoruserid', 'ownerdisplayname', 'owneruserid', 'parentid',
       'posttypeid', 'score', 'tags', 'title', 'viewcount'],
      dtype='str')
Index(['id', 'creationdate', 'postid', 'score', 'userdisplayname', 'userid'], dtype='str')
Index(['id', 'accountid', 'creationdate', 'displayname', 'downvotes',
       'lastaccessdate', 'location', 'reputation', 'upvotes', 'views'],
      dtype='str')


In [4]:
print(post_df.isna().sum(), "\n\n\n")
print(comment_df.isna().sum(), "\n\n\n")
print(user_df.isna().sum())

id                            0
acceptedanswerid         258156
answercount              155036
closeddate               298623
commentcount                  0
communityowneddate       304268
creationdate                  0
favoritecount            257294
lastactivitydate              0
lasteditdate             154479
lasteditordisplayname    305801
lasteditoruserid         157529
ownerdisplayname         301918
owneruserid                4065
parentid                 156603
posttypeid                    0
score                         0
tags                     155036
title                    155036
viewcount                155036
dtype: int64 



id                      0
creationdate            0
postid                  0
score                   0
userdisplayname    559956
userid               8875
dtype: int64 



id                     0
accountid             12
creationdate           0
displayname           16
downvotes              0
lastaccessdate         0
location          16

In [5]:
# user_questioned = 5
# user_answered = 5
# user_commented = 5

df_5_question = post_df[(post_df["posttypeid"] == 1) & (post_df["owneruserid"] == 5)][["id", "posttypeid", "owneruserid"]]
df_5_answer = post_df[(post_df["posttypeid"] == 2) & (post_df["owneruserid"] == 5)][["id", "posttypeid", "owneruserid", "parentid"]]

df_5_comment = comment_df[comment_df["userid"] == 5][["postid", "userid"]]

# Join to get all answers to questions asked by user 5
df_5_joined = df_5_question.merge(df_5_answer, left_on="id", right_on="parentid", how="inner", suffixes=("_question", "_answer"))
df_5_joined

,id_question,posttypeid_question,owneruserid_question,id_answer,posttypeid_answer,owneruserid_answer,parentid
0,103,1,5.0,108,2,5.0,103.0
1,125,1,5.0,126,2,5.0,125.0
2,423,1,5.0,433,2,5.0,423.0
3,423,1,5.0,2217,2,5.0,423.0
4,562,1,5.0,569,2,5.0,562.0
5,1826,1,5.0,1828,2,5.0,1826.0
6,1906,1,5.0,1907,2,5.0,1906.0
7,1908,1,5.0,1910,2,5.0,1908.0


In [6]:
df_5_joined.merge(df_5_comment, left_on="id_answer", right_on="postid", how="inner")

,id_question,posttypeid_question,owneruserid_question,id_answer,posttypeid_answer,owneruserid_answer,parentid,postid,userid
0,1826,1,5.0,1828,2,5.0,1826.0,1828,5.0


# **a) Write a MapReduce job with “posts.csv” and "comment.csv" as inputs and following output:**

        (user_questioned, user_answered, user_commented)

where each triple represents a complete question–answer–comment relationship: a user who asked a question, a user who answered that question, and a user who commented on the corresponding answer.

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). For answer posts, the column "parentid" stores the id of the corresponding question. In "comments.csv", the answer corresponding to the comment is stored in "postid" and the user who commented is stored in "userid" (You may ignore the comments where the "userid" is empty).

The output should begin with the following lines:

```
8,108,6
8,108,2750
8,319,2750
24,145,919
24,145,2669
24,930,1048
24,16188,919
117,61,117
117,61,61
28524,25392,28524
```


- Post.csv

| index | column name           |
| ----- | --------------------- |
| 0     | id                    |
| 1     | acceptedanswerid      |
| 2     | answercount           |
| 3     | closeddate            |
| 4     | commentcount          |
| 5     | communityowneddate    |
| 6     | creationdate          |
| 7     | favoritecount         |
| 8     | lastactivitydate      |
| 9     | lasteditdate          |
| 10    | lasteditordisplayname |
| 11    | lasteditoruserid      |
| 12    | ownerdisplayname      |
| 13    | owneruserid           |
| 14    | parentid              |
| 15    | posttypeid            |
| 16    | score                 |
| 17    | tags                  |
| 18    | title                 |
| 19    | viewcount             |


- Comment.csv

| index | column          |
| ----- | --------------- |
| 0     | id              |
| 1     | creationdate    |
| 2     | postid          |
| 3     | score           |
| 4     | userdisplayname |
| 5     | userid          |



In [7]:
%%file mymrjob1.py

# This will create a local file to run your MapReduce program

from mrjob.job import MRJob
from mrjob.step import MRStep
from mrjob.util import log_to_stream
from mrjob.compat import jobconf_from_env
from mrjob.protocol import RawValueProtocol
from mr3px.csvprotocol import CsvProtocol
import csv
import io

import logging

# 
#  Below is the skeleton for a MapReduce program in mrjob.
#  Write your own solution here. Be sure that it actually runs successfully.


class MyMRJob1(MRJob):
    OUTPUT_PROTOCOL = RawValueProtocol

    POSTS_ID_INDEX = 0  # id
    POSTS_OWNER_INDEX = 13  # owneruserid
    POSTS_PARENT_INDEX = 14  # parentid
    POSTS_POST_TYPE_INDEX = 15  # posttypeid
    COMMENTS_POST_INDEX = 2  # postid
    COMMENTS_USER_INDEX = 5  # userid

    @classmethod
    def set_up_logging(cls, quiet=False, verbose=False, stream=None):
        log_to_stream(name='mrjob', debug=verbose, stream=stream)
        log_to_stream(name='__main__', debug=verbose, stream=stream)

    def mapper(self, _, line):
        row = next(csv.reader([line]))
        if row is None or (row[0] == "id"):
            return  # Skip unparseable lines and header lines

        # This script handles posts.csv
        if len(row) > 15:
            post_id = row[self.POSTS_ID_INDEX].split(".")[0]
            post_type = row[self.POSTS_POST_TYPE_INDEX].split(".")[0]
            owner_user_id = row[self.POSTS_OWNER_INDEX].split(".")[0]
            parent_id = row[self.POSTS_PARENT_INDEX].split(".")[0]

            # This is a question
            if post_type == "1":
                if post_id != "" and owner_user_id != "":
                    yield post_id, ("QUESTION", owner_user_id)\

            # This is an answer
            elif post_type == "2":
                if parent_id != "" and owner_user_id != "" and post_id != "":
                    yield parent_id, ("ANSWER", post_id, owner_user_id)

        # This script handles comments.csv
        else:
            answer_id = row[self.COMMENTS_POST_INDEX].split(".")[0]
            commenter_user_id = row[self.COMMENTS_USER_INDEX].split(".")[0]

            if answer_id != "" and commenter_user_id != "":
                yield answer_id, ("COMMENT", commenter_user_id)

    def reducer_prepare_answers(
        self,
        key,  # post_id (question_id or answer_id)
        values,
    ):

        question_owner = None
        answers = []
        comments = []

        for value in values:
            if value[0] == "QUESTION":  # key is question_id, value is question_owner
                question_owner = value[1]
            elif value[0] == "ANSWER":  # key is answer_id, value is answer_owner
                answer_id = value[1]  # for semantic meaning purposes, value[1] is answer_id
                answer_owner = value[2]
                answers.append((answer_id, answer_owner))
            elif value[0] == "COMMENT":  # key is answer_id, value is commenter_user_id
                comments.append(value[1])

        if question_owner and question_owner != "" and len(answers) > 0:
            for answer_id, answer_owner in answers:
                yield answer_id, ("QUESTION_ANSWER", question_owner, answer_owner)

        for commenter_user_id in comments:  # key is answer_id, value is commenter_user_id
            answer_id = key  # for semantic meaning purposes, key is answer_id
            if commenter_user_id and commenter_user_id != "":
                yield answer_id, ("COMMENTER", commenter_user_id)

    def reducer_emit_triples(self, answer_id, values):
        commenters = set()
        user_questioned = None
        user_answered = None

        for value in values:
            if value[0] == "QUESTION_ANSWER":
                user_questioned = value[1]
                user_answered = value[2]
            elif value[0] == "COMMENTER":
                commenters.add(value[1])

        if user_questioned and user_questioned != "" and user_answered and user_answered != "" and len(commenters) > 0:
            for commenter_user_id in commenters:            
                yield "ONE_KEY", f"{user_questioned},{user_answered},{commenter_user_id}"

    def reducer_sort(self, _, lines):
        all_rows = []
        for line in lines:    
            q, a, c = line.split(",")
            all_rows.append((int(q), int(a), int(c)))

        for line in sorted(all_rows):
            yield None, f"{line[0]},{line[1]},{line[2]}"

    def steps(self):
        return [
            MRStep(
                mapper=self.mapper,
                reducer=self.reducer_prepare_answers,
            ),
            MRStep(reducer=self.reducer_emit_triples),
            MRStep(reducer=self.reducer_sort),
        ]


if __name__ == "__main__":
    MyMRJob1.run()


Overwriting mymrjob1.py


In [8]:
! python3 mymrjob1.py {posts} {comments}

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/mymrjob1.cam-le.20260509.160155.635468
Running step 1 of 3...
Running step 2 of 3...
Running step 3 of 3...
job output is in /tmp/mymrjob1.cam-le.20260509.160155.635468/output
Streaming final output from /tmp/mymrjob1.cam-le.20260509.160155.635468/output...
5,5,5
5,5,196
5,5,582
5,5,1146
5,5,6579
5,5,9007
5,5,11887
5,5,16663
5,5,17359
5,5,69701
5,5,109054
5,5,179103
5,8,264
5,8,35989
5,11,5
5,11,5
5,11,11
5,13,174
5,13,7962
5,13,11887
5,23,23
5,23,30
5,23,59
5,23,74
5,23,81525
5,25,5
5,25,4916
5,25,41284
5,30,223
5,30,919
5,30,5739
5,30,7224
5,30,12615
5,30,36041
5,61,247
5,74,442
5,74,44269
5,159,4908
5,183,183
5,183,334
5,183,1577
5,183,5280
5,183,7365
5,183,9007
5,183,22452
5,183,32036
5,183,44269
5,183,92109
5,190,5
5,190,223
5,247,5
5,251,110
5,253,919
5,253,4925
5,253,5256
5,253,5836
5,253,19788
5,253,62422
5,339,805
5,339,2860
5,339,6244
5,339,9007
5,339,36769

# b) Write a MapReduce job with “posts.csv” and "users.csv" as input and following output:**  

For each user compute the total number of question and answer posts they have created and output the 100 users with highest reputation.

Make sure the output has the following schema:

            displayname, i, reputation, question_count, answer_count

In "posts.csv", the user ids are stored in "owneruserid". Column "posttypeid" stores whether a post is a question (posttypeid = 1) or an answer (posttypeid=2). In "users.csv", the user ids are stored in "id" and their reputations are stored in "reputation".

In case that there are multiple users with the same reputation, you may resolve these conflicts randomly, i.e. pick one of them arbitrarily.

Use a combiner function where possible to reduce the communication cost.

Make sure that your program correctly deals with the header, data types, and possible sparse values.

In case of a correct implementation, the output should contain this text:

```
"Glen_b -Reinstate Monica",1,232889,18,4504
"whuber",2,231500,8,2377
"gung - Reinstate Monica",3,117552,12,1471
"Peter Flom - Reinstate Monica",4,87309,78,2814
"amoeba says Reinstate Monica",5,77182,43,344
```



- Post.csv

| index | column name           |
| ----- | --------------------- |
| 0     | id                    |
| 1     | acceptedanswerid      |
| 2     | answercount           |
| 3     | closeddate            |
| 4     | commentcount          |
| 5     | communityowneddate    |
| 6     | creationdate          |
| 7     | favoritecount         |
| 8     | lastactivitydate      |
| 9     | lasteditdate          |
| 10    | lasteditordisplayname |
| 11    | lasteditoruserid      |
| 12    | ownerdisplayname      |
| 13    | owneruserid           |
| 14    | parentid              |
| 15    | posttypeid            |
| 16    | score                 |
| 17    | tags                  |
| 18    | title                 |
| 19    | viewcount             |

- users.csv

| index | column name           |
| ----- | --------------------- |
| 0     | id                    |
| 1     | accountid             |
| 2     | creationdate          |
| 3     | displayname           |
| 4     | downvotes             |
| 5     | lastaccessdate        |
| 6     | location              |
| 7     | reputation            |
| 8     | upvotes               |
| 9     | views                 |

In [9]:
%%file mymrjob2.py

# This will create a local file to run your MapReduce program

from mrjob.job import MRJob
from mrjob.protocol import RawValueProtocol
from mrjob.step import MRStep
from mrjob.util import log_to_stream
import csv
import heapq
import io


class MyMRJob2(MRJob):
    OUTPUT_PROTOCOL = RawValueProtocol

    TOP_K = 100

    POSTS_ID_INDEX = 0  # id
    POSTS_OWNER_INDEX = 13  # owneruserid
    POSTS_PARENT_INDEX = 14  # parentid
    POSTS_POST_TYPE_INDEX = 15  # posttypeid

    USER_ID_INDEX = 0  # id
    USERS_REPUTATION_INDEX = 7  # reputation
    USERS_DISPLAYNAME_INDEX = 3  # displayname

    @classmethod
    def set_up_logging(cls, quiet=False, verbose=False, stream=None):
        log_to_stream(name='mrjob', debug=verbose, stream=stream)
        log_to_stream(name='__main__', debug=verbose, stream=stream)

    def mapper(self, _, line):
        row = next(csv.reader([line]))
        if row is None or (row[0] == "id"):
            return  # Skip unparseable lines and header lines

        user_id = None
        post_type = None
        reputation = None
        displayname = None

        # This script handles posts.csv
        if len(row) > 15:
            user_id = row[self.POSTS_OWNER_INDEX].split(".")[0].strip()
            post_type = row[self.POSTS_POST_TYPE_INDEX].split(".")[0].strip()

            # To count number of questions/answers for each user
            if post_type == "1":
                yield user_id, ("POST_COUNTS", 1, 0)
            elif post_type == "2":
                yield user_id, ("POST_COUNTS", 0, 1)

        # This script handles users.csv
        else:
            user_id = row[self.USER_ID_INDEX].split(".")[0].strip()
            reputation = row[self.USERS_REPUTATION_INDEX].split(".")[0].strip()
            displayname = row[self.USERS_DISPLAYNAME_INDEX].strip()

            if reputation is None or reputation == "":
                reputation = 0

            if displayname is None or displayname == "":
                displayname = ""

            # To get each user's relevant information
            yield user_id, ("USER", displayname, reputation)


    def combiner(self, user_id, values):
        displayname = ""
        reputation = 0
        question_count = 0
        answer_count = 0

        for value in values:
            if value[0] == "POST_COUNTS":
                question_count += value[1]
                answer_count += value[2]
            elif value[0] == "USER":
                displayname = value[1]
                reputation += int(value[2])

        yield user_id, ("POST_COUNTS", question_count, answer_count)
        yield user_id, ("USER", displayname, reputation)

    def reducer(self, user_id, values):
        displayname = ""
        reputation = 0
        question_count = 0
        answer_count = 0

        for value in values:
            if value[0] == "POST_COUNTS":
                question_count += value[1]
                answer_count += value[2]
            elif value[0] == "USER":
                displayname = value[1]
                reputation += int(value[2])

        yield "ONE_KEY", (user_id, displayname, reputation, question_count, answer_count)

    def reducer_top_k(self, _, values):
        heap = []

        for user_id, displayname, reputation, question_count, answer_count in values:
            candidate = (reputation, user_id, displayname, question_count, answer_count)

            if len(heap) < self.TOP_K:
                heapq.heappush(heap, candidate)
            else:
                heapq.heappushpop(heap, candidate)

        top_users = sorted(heap, key=lambda item: (-item[0], item[1]))

        for reputation, user_id, displayname, question_count, answer_count in top_users:
            yield None, f"{displayname}, {reputation}, {question_count}, {answer_count}"

    def steps(self):
        return [
            MRStep(
                mapper=self.mapper,
                combiner=self.combiner,
                reducer=self.reducer,
            ),
            MRStep(
                reducer=self.reducer_top_k,
            ),
        ]


if __name__ == "__main__":
    MyMRJob2.run()


Overwriting mymrjob2.py


In [10]:
!python3 mymrjob2.py {posts} {users}

No configs found; falling back on auto-configuration
No configs specified for inline runner
Creating temp directory /tmp/mymrjob2.cam-le.20260509.160202.413049
Running step 1 of 2...
Running step 2 of 2...
job output is in /tmp/mymrjob2.cam-le.20260509.160202.413049/output
Streaming final output from /tmp/mymrjob2.cam-le.20260509.160202.413049/output...
Glen_b -Reinstate Monica, 232889, 18, 4504
whuber, 231500, 8, 2377
gung - Reinstate Monica, 117552, 12, 1471
Peter Flom - Reinstate Monica, 87309, 78, 2814
amoeba says Reinstate Monica, 77182, 43, 344
Tim, 74796, 66, 1399
Xi'an, 68814, 15, 1324
Frank Harrell, 60786, 16, 1075
Stephan Kolassa, 60724, 15, 1303
Sycorax says Reinstate Monica, 53606, 31, 845
Ben - Reinstate Monica, 47684, 26, 1165
chl, 46994, 8, 437
Alecos Papadopoulos, 46869, 40, 862
Rob Hyndman, 44883, 8, 384
Aksakal, 44248, 33, 1290
ttnphns, 44117, 25, 389
AdamO, 42871, 91, 1371
Nick Cox, 42209, 0, 775
kjetil b halvorsen, 42026, 12, 1579
Greg Snow, 41868, 0, 823
Jeromy Ang

---
## **Your solution for Exercise 4 will consist of:**  
*  This notebook, filled with your solutions. 
